## The goal is to find revenue growth opportunities
## Based on EDA

In [144]:
import pandas as pd
import sqlite3

In [145]:
conn = sqlite3.connect('../data/database/data_mart.db') 

## Problem 3: Revenue concentration in high-value orders

---

## Hypothesis 1: Revenue is driven by high-value customers

In [146]:
pd.read_sql("SELECT * FROM data_mart", conn)

,order_id,order_item_id,product_id,customer_id,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,price,freight_value,revenue,product_category_name,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,2017-09-13 08:59:02,2017-09-20 23:43:48,58.90,13.29,72.19,cool_stuff,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,2017-04-26 10:53:06,2017-05-12 16:04:24,239.90,19.93,259.83,pet_shop,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,2018-01-14 14:33:31,2018-01-22 13:19:16,199.00,17.87,216.87,moveis_decoracao,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,2018-08-08 10:00:35,2018-08-14 13:32:39,12.99,12.79,25.78,perfumaria,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,2017-02-04 13:57:51,2017-03-01 16:42:31,199.90,18.14,218.04,ferramentas_jardim,varzea paulista,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...
112275,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b51593916b4b8e0d6f66f2ae24f2673d,0c9aeda10a71f369396d0c04dce13a64,2018-04-23 13:57:06,2018-05-10 22:56:40,299.99,43.41,343.40,utilidades_domesticas,sao luis,MA
112276,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,84c5d4fbaf120aae381fad077416eaa0,0da9fe112eae0c74d3ba1fe16de0988b,2018-07-14 10:26:46,2018-07-23 20:31:55,350.00,36.53,386.53,informatica_acessorios,curitiba,PR
112277,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,29309aa813182aaddc9b259e31b870e6,cd79b407828f02fdbba457111c38e4c4,2017-10-23 17:07:56,2017-10-28 12:22:22,99.90,16.95,116.85,esporte_lazer,sao paulo,SP
112278,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,b5e6afd5a41800fdf401e0272ca74655,eb803377c9315b564bdedad672039306,2017-08-14 23:02:59,2017-08-16 21:59:40,55.99,8.72,64.71,informatica_acessorios,vinhedo,SP


In [147]:
rfm_customer = pd.read_sql("SELECT * FROM data_mart", conn).groupby('customer_unique_id').agg({
    'revenue':'sum',
    'order_purchase_timestamp':'max',
    'order_id':'nunique'
}).reset_index()
rfm_customer['order_purchase_timestamp'] = pd.to_datetime(rfm_customer['order_purchase_timestamp'])
reference_date = rfm_customer['order_purchase_timestamp'].max()
rfm_customer['recency'] = (reference_date - rfm_customer['order_purchase_timestamp']).dt.days

In [148]:
rfm_customer['R'] = pd.qcut(rfm_customer['recency'], 5, labels=[5,4,3,2,1])
def frequency_segment(x):
    if x == 1:
        return '1_order'
    elif x <= 3:
        return '2-3_orders'
    else:
        return '4+_orders'
rfm_customer['F'] = rfm_customer['order_id'].apply(frequency_segment)
rfm_customer['M'] = pd.qcut(rfm_customer['revenue'], 5, labels=[1,2,3,4,5])
rfm_customer.to_sql('rfm_customers',conn,if_exists='replace',index=False)

95121

#### VIP/high value segment

In [149]:
VIP_segment = pd.read_sql(
    """
    SELECT *
    FROM rfm_customers
    WHERE M IN (4, 5) AND F IN ('2-3_orders', '4+_orders') AND R IN (4, 5)
    """,conn)
VIP_segment

,customer_unique_id,revenue,order_purchase_timestamp,order_id,recency,R,F,M
0,004b45ec5c64187465168251cd1c9c2f,147.72,2018-05-26 19:42:48,2,94,4,2-3_orders,4
1,0058f300f57d7b93c477a131a59b36c3,175.58,2018-03-22 18:09:41,2,159,4,2-3_orders,4
2,011575986092c30523ecb71ff10cb473,214.90,2018-04-18 21:58:08,2,132,4,2-3_orders,5
3,012452d40dafae4df401bced74cdb490,495.33,2018-05-14 12:12:45,2,107,4,2-3_orders,5
4,012a218df8995d3ec3bb221828360c86,1510.38,2018-06-18 13:08:38,2,72,5,2-3_orders,5
...,...,...,...,...,...,...,...,...
1011,fd8ccc89be43894d2553494c71a61fd8,258.03,2018-04-19 08:19:39,3,132,4,2-3_orders,5
1012,fe3e52de024b82706717c38c8e183084,157.30,2018-07-31 08:30:27,2,29,5,2-3_orders,4
1013,fe81bb32c243a86b2f86fbf053fe6140,1590.76,2018-06-21 12:10:25,5,69,5,4+_orders,5
1014,fed519569d16e690df6f89cb99d4e682,286.14,2018-03-18 21:51:49,2,163,4,2-3_orders,5


In [150]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS count_VIP,
        COUNT(*) / CAST((SELECT COUNT(*) FROM rfm_customers) AS float)  AS percent_from_total_count_VIP,
        SUM(revenue) AS sum_revenue_VIP,
        SUM(revenue) / CAST((SELECT SUM(revenue) FROM rfm_customers) AS float)  AS percent_from_total_revenue_VIP
    FROM rfm_customers
    WHERE M IN (4, 5) AND F IN ('2-3_orders', '4+_orders') AND R IN (4, 5)
    """,conn)

,count_VIP,percent_from_total_count_VIP,sum_revenue_VIP,percent_from_total_revenue_VIP
0,1016,0.010681,378957.23,0.024006


#### Loyal customers segment (high-ticket)

In [151]:
loyal_high_segment = pd.read_sql(
    """
    SELECT *
    FROM rfm_customers
    WHERE M IN (2, 3) AND F IN ('2-3_orders', '4+_orders') AND R IN (4, 5)
    """,conn)
loyal_high_segment

,customer_unique_id,revenue,order_purchase_timestamp,order_id,recency,R,F,M
0,00a39521eb40f7012db50455bf083460,123.25,2018-06-03 10:12:57,2,87,5,2-3_orders,3
1,01886ef98f995e4f2dd75a1d04c97397,75.31,2018-03-09 19:19:24,2,172,4,2-3_orders,2
2,02e9827c15167f699df7bf90d326ffdb,68.86,2018-04-07 18:22:27,2,143,4,2-3_orders,2
3,03d4953a455f8489e712ea80699de878,72.65,2018-07-21 15:45:49,2,38,5,2-3_orders,2
4,04fc637963fe982dd25ac58e4f4770c1,127.59,2018-03-25 21:20:12,2,156,4,2-3_orders,3
...,...,...,...,...,...,...,...,...
211,f97ab9c166ca8d6f5313653ee439a6a8,66.28,2018-08-20 15:37:16,2,8,5,2-3_orders,2
212,fad2bf8fa1bdfc125781eb6c62b5ad81,86.36,2018-04-10 07:23:55,2,141,4,2-3_orders,2
213,fb920961e4d8e55fd9c357976a611765,66.55,2018-06-24 10:09:52,2,66,5,2-3_orders,2
214,fcd0ab79592faab19e2bf386cf69fbcd,113.08,2018-04-07 21:18:46,2,143,4,2-3_orders,3


In [152]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS count_loyal_high,
        COUNT(*) / CAST((SELECT COUNT(*) FROM rfm_customers) AS float)  AS percent_from_total_count_loyal_high,
        SUM(revenue) AS sum_revenue_loyal_high,
        SUM(revenue) / CAST((SELECT SUM(revenue) FROM rfm_customers) AS float)  AS percent_from_total_revenue_loyal_high
    FROM rfm_customers
    WHERE M IN (2, 3) AND F IN ('2-3_orders', '4+_orders') AND R IN (4, 5)
    """,conn)

,count_loyal_high,percent_from_total_count_loyal_high,sum_revenue_loyal_high,percent_from_total_revenue_loyal_high
0,216,0.002271,21741.09,0.001377


#### Loyal customer segment (low-ticket)

In [153]:
loyal_low_segment = pd.read_sql(
    """
    SELECT *
    FROM rfm_customers
    WHERE M == 1 AND F IN ('2-3_orders', '4+_orders') AND R IN (4, 5)
    """,conn)
loyal_low_segment

,customer_unique_id,revenue,order_purchase_timestamp,order_id,recency,R,F,M
0,031e19fc630c4121f1238716f41675c3,35.94,2018-07-04 01:20:01,2,56,5,2-3_orders,1
1,08fb46d35bb3ab4037202c23592d1259,44.72,2018-06-04 16:44:48,2,85,5,2-3_orders,1
2,4186b96df8197e7b4982a751c1dde3b6,55.08,2018-08-16 18:16:43,2,12,5,2-3_orders,1
3,4a6ccb379c9e5062ad3ce2324222fbf3,42.47,2018-03-28 17:03:47,2,153,4,2-3_orders,1
4,55fbb64f60861980e74c4f136f304297,44.93,2018-08-04 17:54:31,2,24,5,2-3_orders,1
5,6e2f539e9e0e323859a81079581db712,54.15,2018-04-03 15:26:14,2,147,4,2-3_orders,1
6,6f5c52ea47e32b73958b0ac0f3c34e88,52.13,2018-05-27 12:20:21,2,94,4,2-3_orders,1
7,7b6d07dac0a2c373d749142e920ae356,42.47,2018-04-09 12:36:56,2,142,4,2-3_orders,1
8,91e59b92e87f627abdab279ea3ab0c87,38.78,2018-03-31 13:07:07,2,151,4,2-3_orders,1
9,ab6ec178286e6a6d70e9bfe90378bd95,39.18,2018-06-20 10:59:40,2,70,5,2-3_orders,1


In [154]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS count_loyal_low,
        COUNT(*) / CAST((SELECT COUNT(*) FROM rfm_customers) AS float)  AS percent_from_total_count_loyal_low,
        SUM(revenue) AS sum_revenue_loyal_low,
        SUM(revenue) / CAST((SELECT SUM(revenue) FROM rfm_customers) AS float)  AS percent_from_total_revenue_loyal_low
    FROM rfm_customers
    WHERE M == 1 AND F IN ('2-3_orders', '4+_orders') AND R IN (4, 5)
    """,conn)

,count_loyal_low,percent_from_total_count_loyal_low,sum_revenue_loyal_low,percent_from_total_revenue_loyal_low
0,16,0.000168,711.09,0.000045


#### Segment of one-time clients (active)

In [155]:
one_time_active_segment = pd.read_sql(
    """
    SELECT *
    FROM rfm_customers
    WHERE R IN (4, 5) AND F == '1_order'
    """,conn)
one_time_active_segment

,customer_unique_id,revenue,order_purchase_timestamp,order_id,recency,R,F,M
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,2018-05-10 10:56:27,1,111,4,1_order,4
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,2018-05-07 11:11:27,1,114,4,1_order,1
2,0004bd2a26a76fe21f786e4fbd80607f,166.98,2018-04-05 19:33:16,1,145,4,1_order,4
3,00050ab1314c0e55a6ca13cf7181fecf,35.38,2018-04-20 12:57:23,1,131,4,1_order,1
4,0005ef4cd20d2893f0d9fbd94d3c0d97,129.76,2018-03-12 15:22:12,1,169,4,1_order,3
...,...,...,...,...,...,...,...,...
36828,fff3e1d7bc75f11dc7670619b2e61840,82.51,2018-07-20 13:47:30,1,40,5,1_order,2
36829,fff5eb4918b2bf4b2da476788d42051c,2844.96,2018-07-02 16:39:59,1,57,5,1_order,5
36830,fff96bc586f78b1f070da28c4977e810,63.42,2018-08-15 10:26:57,1,14,5,1_order,2
36831,fffcc512b7dfecaffd80f13614af1d16,710.70,2018-04-11 00:34:32,1,140,4,1_order,5


In [156]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS count_onetimeactive,
        COUNT(*) / CAST((SELECT COUNT(*) FROM rfm_customers) AS float)  AS percent_from_total_count_onetimeactive,
        SUM(revenue) AS sum_revenue_onetimeactive,
        SUM(revenue) / CAST((SELECT SUM(revenue) FROM rfm_customers) AS float)  AS percent_from_total_revenue_onetimeactive
    FROM rfm_customers
    WHERE R IN (4, 5) AND F == '1_order'
    """,conn)

,count_onetimeactive,percent_from_total_count_onetimeactive,sum_revenue_onetimeactive,percent_from_total_revenue_onetimeactive
0,36833,0.387223,6080838.84,0.385199


#### Segment of one-time clients (dormant)

In [157]:
one_time_sleep_segment = pd.read_sql(
    """
    SELECT *
    FROM rfm_customers
    WHERE R IN (1, 2) AND F == '1_order'
    """,conn)
one_time_sleep_segment

,customer_unique_id,revenue,order_purchase_timestamp,order_id,recency,R,F,M
0,0000f46a3911fa3c0805444483337064,86.22,2017-03-10 21:05:03,1,536,1,1_order,2
1,0000f6ccb0745a6a4b88665a16c9f078,43.62,2017-10-12 20:29:41,1,320,2,1_order,1
2,0004aac84e0df4da2b147fca70cf8255,196.89,2017-11-14 19:45:42,1,287,2,1_order,4
3,0005e1862207bf6ccc02e4228effd9a0,150.12,2017-03-04 23:32:12,1,542,1,1_order,4
4,0006fdc98a402fceb4eb0ee528f6a8d4,29.00,2017-07-18 09:23:10,1,407,1,1_order,1
...,...,...,...,...,...,...,...,...
36902,fff699c184bcc967d62fa2c6171765f7,55.00,2017-09-01 17:06:54,1,361,2,1_order,1
36903,fffa431dd3fcdefea4b1777d114144f2,81.20,2017-10-30 20:39:50,1,302,2,1_order,2
36904,fffcf5a5ff07b0908bd4e2dbc735a684,2067.42,2017-06-08 21:00:36,1,446,1,1_order,5
36905,ffff371b4d645b6ecea244b27531430a,112.46,2017-02-07 15:49:16,1,567,1,1_order,3


In [ ]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS count_onetimedormant,
        COUNT(*) / CAST((SELECT COUNT(*) FROM rfm_customers) AS float)  AS percent_from_total_count_onetimedormant,
        SUM(revenue) AS sum_revenue_count_onetimedormant,
        SUM(revenue) / CAST((SELECT SUM(revenue) FROM rfm_customers) AS float)  AS percent_from_total_revenue_count_onetimedormant
    FROM rfm_customers
    WHERE R IN (1, 2) AND F == '1_order'
    """,conn)

,count_onetimesleep,percent_from_total_count_onetimesleep,sum_revenue_onetimesleep,percent_from_total_revenue_onetimesleep
0,36907,0.388001,5999001.02,0.380015


#### Medium-sized customer segment

In [159]:
middle_segment = pd.read_sql(
    """
    SELECT *
    FROM rfm_customers
    WHERE R == 3
    """,conn)
middle_segment

,customer_unique_id,revenue,order_purchase_timestamp,order_id,recency,R,F,M
0,00053a61a98854899e70ed204dd4bafe,419.18,2018-02-28 11:15:41,1,182,3,1_order,5
1,000c8bdb58a29e7115cfc257230fb21b,29.00,2017-12-12 22:53:35,1,259,3,1_order,1
2,000d460961d6dbfa3ec6c9f5805769e1,36.68,2018-01-07 22:59:25,1,233,3,1_order,1
3,0014a5a58da615f7b01a4f5e194bf5ea,99.82,2018-01-18 20:49:36,1,222,3,1_order,3
4,00196fdb2bf9edfc35e88ebfbcf8d781,27.00,2018-02-22 19:06:44,1,187,3,1_order,1
...,...,...,...,...,...,...,...,...
19097,fff1bdd5c5e37ca79dd74deeb91aa5b6,172.98,2018-02-24 17:38:14,1,185,3,1_order,4
19098,fff7219c86179ca6441b8f37823ba3d3,265.80,2017-12-27 18:57:38,1,244,3,1_order,5
19099,fffb09418989a0dbff854a28163e47c6,73.16,2017-12-17 19:14:35,1,254,3,1_order,2
19100,fffbf87b7a1a6fa8b03f081c5f51a201,167.32,2017-12-27 22:36:41,1,244,3,1_order,4


In [160]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS count_middle,
        COUNT(*) / CAST((SELECT COUNT(*) FROM rfm_customers) AS float)  AS percent_from_total_count_middle,
        SUM(revenue) AS sum_revenue_middle,
        SUM(revenue) / CAST((SELECT SUM(revenue) FROM rfm_customers) AS float)  AS percent_from_total_revenue_middle
    FROM rfm_customers
    WHERE R == 3
    """,conn)

,count_middle,percent_from_total_count_middle,sum_revenue_middle,percent_from_total_revenue_middle
0,19102,0.200818,2996264.0,0.189802


#### Lost Customer Segment

In [161]:
lost_segment = pd.read_sql(
    """
    SELECT *
    FROM rfm_customers
    WHERE R IN (1, 2) AND F IN ('2-3_orders', '4+_orders')
    """,conn)
lost_segment

,customer_unique_id,revenue,order_purchase_timestamp,order_id,recency,R,F,M
0,00cc12a6d8b578b8ebd21ea4e2ae8b27,126.20,2017-03-21 19:25:23,2,525,1,2-3_orders,3
1,013f4353d26bb05dc6652f1269458d8d,356.39,2017-11-28 13:30:58,2,274,2,2-3_orders,5
2,015557c9912277312b9073947804a7ba,315.12,2017-05-01 14:48:33,2,485,1,2-3_orders,5
3,018b5a7502c30eb5f230f1b4eb23a156,110.18,2017-08-20 18:10:06,2,373,2,2-3_orders,3
4,02168ea18740a0fdaaa15f11bebba5db,264.04,2017-10-09 22:05:59,2,323,2,2-3_orders,5
...,...,...,...,...,...,...,...,...
1026,fed2005ccab4fcf1a40ebdaff032a148,101.10,2017-06-17 17:04:31,2,437,1,2-3_orders,3
1027,ff44401d0d8f5b9c54a47374eb48c1b8,68.00,2017-05-19 21:20:54,2,466,1,2-3_orders,2
1028,ff8892f7c26aa0446da53d01b18df463,330.14,2017-11-26 23:25:43,2,275,2,2-3_orders,5
1029,ff922bdd6bafcdf99cb90d7f39cea5b3,139.60,2017-09-14 14:24:04,3,349,2,2-3_orders,4


In [162]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS count_lost,
        COUNT(*) / CAST((SELECT COUNT(*) FROM rfm_customers) AS float)  AS percent_from_total_count_lost,
        SUM(revenue) AS sum_revenue_lost,
        SUM(revenue) / CAST((SELECT SUM(revenue) FROM rfm_customers) AS float)  AS percent_from_total_revenue_lost
    FROM rfm_customers
    WHERE R IN (1, 2) AND F IN ('2-3_orders', '4+_orders')
    """,conn)

,count_lost,percent_from_total_count_lost,sum_revenue_lost,percent_from_total_revenue_lost
0,1031,0.010839,308709.92,0.019556


**Result:**
- Top 5% of orders generate about 34% of revenue  
- Majority of customers are one-time buyers  
- No significant loyal/VIP segment (RFM)  

**Conclusion:** Not confirmed  
Revenue is not driven by a stable high-value customer base

---

## Hypothesis 2: High-value orders are driven by product factors


In [163]:
df_data_mart_short = pd.read_sql(
    """
    SELECT 
        order_id,
        product_category_name,
        revenue
    FROM data_mart
    WHERE product_category_name IN (
        SELECT product_category_name
        FROM data_mart
        GROUP BY product_category_name
        HAVING COUNT(order_id) >= 1000)
    """,conn)
df_data_mart_short

,order_id,product_category_name,revenue
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,72.19
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,259.83
2,000229ec398224ef6ca0657da4fc703e,moveis_decoracao,216.87
3,00024acbcdf0a6daa1e931b038114c75,perfumaria,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,ferramentas_jardim,218.04
...,...,...,...
100518,fffc94f6ce00a00581880bf54a75a037,utilidades_domesticas,343.40
100519,fffcd46ef2263f404302a634eb57f7eb,informatica_acessorios,386.53
100520,fffce4705a9662cd70adb13d4a31832d,esporte_lazer,116.85
100521,fffe18544ffabc95dfada21779c9644f,informatica_acessorios,64.71


In [164]:
df_top_orders = df_data_mart_short[df_data_mart_short['revenue'] > df_data_mart_short['revenue'].quantile(0.95)]
df_top_orders['product_category_name'].value_counts() / df_top_orders.shape[0]

product_category_name
beleza_saude                   0.171275
relogios_presentes             0.156356
automotivo                     0.072608
esporte_lazer                  0.066640
cool_stuff                     0.065844
informatica_acessorios         0.056893
ferramentas_jardim             0.046748
utilidades_domesticas          0.046748
brinquedos                     0.046350
bebes                          0.045355
perfumaria                     0.038194
moveis_decoracao               0.027452
consoles_games                 0.026059
cama_mesa_banho                0.025661
moveis_escritorio              0.023672
telefonia                      0.022678
pet_shop                       0.013925
eletronicos                    0.012333
Unknown                        0.010941
fashion_bolsas_e_acessorios    0.009747
malas_acessorios               0.007559
papelaria                      0.006962
Name: count, dtype: float64

In [165]:
df_data_mart_short['product_category_name'].value_counts() / df_data_mart_short.shape[0]

product_category_name
cama_mesa_banho                0.110492
beleza_saude                   0.095690
esporte_lazer                  0.085771
moveis_decoracao               0.082220
informatica_acessorios         0.077654
utilidades_domesticas          0.069158
relogios_presentes             0.059549
telefonia                      0.045104
ferramentas_jardim             0.043194
automotivo                     0.042010
brinquedos                     0.040687
cool_stuff                     0.037683
perfumaria                     0.033684
bebes                          0.030351
eletronicos                    0.027506
papelaria                      0.025039
fashion_bolsas_e_acessorios    0.020115
pet_shop                       0.019329
moveis_escritorio              0.016762
Unknown                        0.015927
consoles_games                 0.011211
malas_acessorios               0.010863
Name: count, dtype: float64

In [166]:
((df_top_orders['product_category_name'].value_counts() / df_top_orders.shape[0]) / (df_data_mart_short['product_category_name'].value_counts() / df_data_mart_short.shape[0])).sort_values(ascending=False)

product_category_name
relogios_presentes             2.625684
consoles_games                 2.324363
beleza_saude                   1.789904
cool_stuff                     1.747329
automotivo                     1.728337
bebes                          1.494339
moveis_escritorio              1.412224
brinquedos                     1.139172
perfumaria                     1.133890
ferramentas_jardim             1.082267
esporte_lazer                  0.776950
informatica_acessorios         0.732646
pet_shop                       0.720413
malas_acessorios               0.695853
Unknown                        0.686954
utilidades_domesticas          0.675950
telefonia                      0.502782
fashion_bolsas_e_acessorios    0.484587
eletronicos                    0.448387
moveis_decoracao               0.333882
papelaria                      0.278062
cama_mesa_banho                0.232247
Name: count, dtype: float64

**Result:**

High-value orders are disproportionately concentrated in specific product categories.

Top categories such as:
- relogios_presentes (2,6x)
- consoles_games (2,3x)
- beleza_saude (1,8x)

are significantly overrepresented in the top 5% of orders compared to their baseline share.

This indicates that certain categories have a much higher likelihood of generating high-value transactions.

**Conclusion:** confirmed. Product category is a key driver of high-value orders. A small group of categories contributes disproportionately to high-revenue transactions.

In [ ]:
conn.close()